In [1]:
import pandas as pd, numpy as np, joblib
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score as ari
from scipy.stats import spearmanr

veri = pd.read_csv("../data/clean/countries_clean.csv")
ulkeler = pd.read_csv("../data/clean/ulkeler_1m.csv")
paket = joblib.load("../models/kume_modeli.joblib")
F = paket["ozellikler"]

gecmis = veri[veri.Time <= 2023].copy()            # 237 ülke (1M eşiği varyantı için)
gecmis["logIMR"] = np.log(gecmis.IMR)
alt, ust = gecmis[gecmis.ISO3_code.isin(ulkeler.iso3)].CNMR.quantile([0.01, 0.99])
gecmis["CNMR_k"] = gecmis.CNMR.clip(alt, ust)

d = gecmis[gecmis.ISO3_code.isin(ulkeler.iso3)].copy()   # 161 ülke
d["temel"] = pd.Series(paket["model"].predict(paket["olcekleyici"].transform(d[F])),
                       index=d.index).map(paket["etiket"])
X = StandardScaler().fit_transform(d[F])
print(d.shape)

(11914, 71)


In [2]:
def sirala(df, etiketler):
    s = pd.Series(etiketler, index=df.index)
    o = df.assign(e=s).groupby("e").TFR.mean().sort_values(ascending=False).index
    return s.map({e: i + 1 for i, e in enumerate(o)})

def bulgular(df, kume):
    df = df.assign(k=kume)
    k_max = df.k.max()
    uc = df[df.Time.isin([1990, 2023])].pivot(index="ISO3_code", columns="Time", values="k")
    tr = df[df.ISO3_code == "TUR"].set_index("Time").k
    return {"1950_ilk_basamak": int((df[df.Time == 1950].k == 1).sum()),
            "2023_ilk_basamak": int((df[df.Time == 2023].k == 1).sum()),
            "1990_2023_geri": int((uc[2023] < uc[1990]).sum()),
            "TUR_2023_en_ust": bool(tr.loc[2023] == k_max),
            "TUR_tum_basamaklar": bool(tr.nunique() == k_max),
            "TUR_geri_adim": bool((tr.sort_index().diff() < 0).any())}

bulgular(d, d.temel)

{'1950_ilk_basamak': 93,
 '2023_ilk_basamak': 0,
 '1990_2023_geri': 0,
 'TUR_2023_en_ust': True,
 'TUR_tum_basamaklar': True,
 'TUR_geri_adim': False}

In [3]:
for s in [1, 7, 99, 2024]:
    e = KMeans(n_clusters=5, random_state=s, n_init=50).fit_predict(X)
    print("random_state", s, "→ ARI:", round(ari(d.temel, e), 3))

random_state 1 → ARI: 0.979
random_state 7 → ARI: 0.987
random_state 99 → ARI: 0.99
random_state 2024 → ARI: 0.998


In [4]:
w = sirala(d, AgglomerativeClustering(n_clusters=5, linkage="ward").fit_predict(X))
print("ARI:", round(ari(d.temel, w), 3), "| Spearman:", round(spearmanr(d.temel, w).correlation, 3))
pd.crosstab(d.temel, w, rownames=["K-means"], colnames=["Ward"])

ARI: 0.53 | Spearman: 0.95


Ward,1,2,3,4,5
K-means,,,,,
1,1832,530,0,0,0
2,0,1899,769,0,0
3,0,16,1367,908,0
4,0,0,2,1965,876
5,0,0,0,0,1750


In [5]:
sonuc = []

def varyant(ad, df, ozellik, k):
    Xv = StandardScaler().fit_transform(df[ozellik])
    e = sirala(df, KMeans(n_clusters=k, random_state=42, n_init=50).fit_predict(Xv))
    ortak = df.ISO3_code.isin(ulkeler.iso3)          # karşılaştırma hep aynı 161 ülkede
    r = {"varyant": ad, "ARI_temel": round(ari(d.temel.loc[df.index[ortak]], e[ortak]), 3)}
    r.update(bulgular(df[ortak], e[ortak]))
    sonuc.append(r)

varyant("Temel (k=5)", d, F, 5)
varyant("k=4", d, F, 4)
varyant("k=6", d, F, 6)
varyant("+ logIMR", d, F + ["logIMR"], 5)
varyant("+ göç", d, F + ["CNMR_k"], 5)
varyant("237 ülke", gecmis, F, 5)

saglamlik = pd.DataFrame(sonuc)
saglamlik

,varyant,ARI_temel,1950_ilk_basamak,2023_ilk_basamak,1990_2023_geri,TUR_2023_en_ust,TUR_tum_basamaklar,TUR_geri_adim
0,Temel (k=5),1.000,93,0,0,True,True,False
1,k=4,0.502,101,0,0,True,True,False
2,k=6,0.540,87,0,0,False,False,False
3,+ logIMR,0.748,98,0,0,True,True,False
4,+ göç,0.480,99,0,2,True,False,False
5,237 ülke,0.824,99,0,0,False,False,False


In [6]:
rng = np.random.default_rng(42)
iso = d.ISO3_code.unique()
skor = []
for b in range(30):
    secim = rng.choice(iso, len(iso), replace=True)
    idx = np.concatenate([np.where(d.ISO3_code.values == i)[0] for i in secim])
    km = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X[idx])
    skor.append(ari(d.temel, km.predict(X)))

print("Bootstrap ARI: ortalama", round(np.mean(skor), 3),
      "| en düşük", round(np.min(skor), 3), "| %5", round(np.quantile(skor, 0.05), 3))

Bootstrap ARI: ortalama 0.785 | en düşük 0.538 | %5 0.625


In [7]:
saglamlik.to_csv("../data/out/saglamlik_kumeleme.csv", index=False)
print("Kaydedildi.")

Kaydedildi.
